Problem Statement: CPQ Decision Roundtable

Context:
- You are building a multi-persona LLM utility that helps consultants pressure-test ambiguous CPQ design decisions by simulating a panel review. The goal is to practice system prompt engineering, the response object, and a basic two-stage LLM chain — all using only the OpenAI Chat Completions API covered on Day 1.

Objective:
- Given a single CPQ/Revenue Cloud design question, generate four distinct expert perspectives in parallel, then produce a synthesized recommendation that reconciles them.

Functional Requirements:

1. Accept a design question as a single string input (hardcoded variable is fine).
2. Define four expert personas as system prompts, each with a clearly differentiated viewpoint, bias, and evaluation lens. The personas must be specific enough that their outputs are visibly distinct from one another.
3. For each persona, generate a response to the design question and display it rendered as Markdown, labeled with the persona name.
4. After collecting all four responses, invoke a fifth "Synthesizer" persona whose user message is dynamically constructed from the four panel responses. The synthesizer must produce a single final recommendation with reasoning that explicitly references the trade-offs surfaced by the panel.
5. Print a token usage summary at the end: per-call tokens for all five calls, plus a grand total.

In [31]:
#All imports
from openai import OpenAI
from IPython.display import Markdown, display


In [32]:
def intialize_openai_client():
    #Load environment variables
    from dotenv import load_dotenv
    load_dotenv()
    import os
    api_key = os.getenv("OPENAI_API_KEY")
    if api_key is None:
        print("Error: OPENAI_API_KEY not found in environment variables.")
        return None
    elif not api_key.startswith("sk-") and len(api_key) < 40:
        print("Error: OPENAI_API_KEY is not valid.")
        return None
    else: 
        print("OPENAI_API_KEY loaded successfully.")
        return api_key

In [33]:
api_key = intialize_openai_client()

OPENAI_API_KEY loaded successfully.


In [34]:
system_prompt_actors = {
    "CPQ Purist": "15 years in classic Salesforce CPQ, biased toward proven patterns, cautious about bleeding edge.",
    "The Revenue Cloud Evangelist": "pushes the new platform, sees CML and decision tables as the future.",
    "The Skeptical CFO": "only cares about TCO, license cost, ROI timeline, and risk.",
    "The End-User Sales Rep": "only cares about clicks-to-quote, training burden, and what breaks their daily flow."
}

user_prompt = "For a manufacturing client selling configurable industrial equipment with 200+ options, should we model this as a Salesforce CPQ bundle with options, or as a Product Configurator pattern using Revenue Cloud's CML-based pricing?"

In [35]:
#We would ask this question to all the actors and get their perspectives. This is a common technique in LLM engineering called "Role Prompting" where we ask the model to take on different roles to get diverse perspectives and introduce bias on a problem.
#We will also take put all the tokens and store it into a dict for each actor. That way we can analyze the token usage for each actor and see which one is more verbose and which one is more concise.
#We will also put each actor's response into a dict so we can analyze the responses later and use them for the final synthesis step.

response_dict = {}
actor_token_usage = {}
for actor, system_prompt in system_prompt_actors.items():
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    openai_client = OpenAI(api_key=api_key)
    response = openai_client.chat.completions.create(model="gpt-4o-mini", messages=messages)
    actor_response = response.choices[0].message.content
    response_dict[actor] = actor_response
    actor_token_usage[actor] = response.usage.total_tokens
    print(f"{actor} and their response:\n")
    display(Markdown(actor_response))

print("Token usage for each actor:")
for actor, token_usage in actor_token_usage.items():
    print(f"{actor}: {token_usage} tokens")

CPQ Purist and their response:



When deciding between modeling the manufacturing client's configurable industrial equipment in Salesforce CPQ as a bundle with options or adopting the Product Configurator pattern using Revenue Cloud's CML (Configurable Margins and Levels)-based pricing, consider the following:

### Salesforce CPQ Bundle with Options:
1. **Simplicity**: Bundles with options can be easier to set up and manage, especially if you have a limited number of complex configurations or if configurations typically follow predictable patterns.
2. **Predefined Structures**: If your equipment can logically be grouped into bundles (e.g., base model + options), CPQ's bundle structure allows a straightforward way to manage this by grouping related products.
3. **Familiarity**: With your extensive experience in Salesforce CPQ, leveraging its features like guided selling, pricing rules, and discounting becomes more straightforward when using bundles.
4. **Performance**: For 200+ options, a well-structured bundle can improve performance for quoting by limiting the combination of options to relevant choices.

### Product Configurator Pattern (CML-based pricing):
1. **Complexity and Customization**: If the configurations are highly customizable based on various parameters and require different pricing strategies, the CML-based pricing model might be more appropriate. It caters to advanced configuration and margin management.
2. **Flexible Pricing**: CML allows for more granular control of pricing, especially in complex scenarios where costs and margins might vary significantly based on selected options and features.
3. **Scalability**: If you anticipate significant growth or an increase in complexity in the future, the Product Configurator approach might allow for easier adjustments and scaling.

### Recommendation:
- **Use a Bundle with Options** if your primary goal is to keep the setup manageable and leverage established best practices with your team's current expertise. It's a familiar pattern that should meet most needs unless future growth for complexity is a concern.
- If you're facing increasingly demanding pricing strategies or if each configuration significantly alters costs, consider the Product Configurator approach with CML-based pricing for its flexibility and depth.

Ultimately, assess the immediate needs versus long-term growth and complexity potential. If possible, engage stakeholders to understand their vision for future expansions and flexibility requirements before making a decision.

The Revenue Cloud Evangelist and their response:



When deciding whether to model configurable industrial equipment with 200+ options as a Salesforce CPQ bundle with options or utilize a Product Configurator pattern using Revenue Cloud's CML (Configure, Model, and Launch)-based pricing, several factors need to be considered:

### Salesforce CPQ Bundle with Options
1. **Simplicity and Usability**: If the configuration process can be simplified with a straightforward choice of options, using CPQ bundles may provide a more user-friendly experience for the sales team and customers.
2. **Integrated Quoting**: Salesforce CPQ is designed for quick quotes, discount management, and approvals, making it easier to manage pricing and proposals.
3. **Standard Product Relationships**: If options have straightforward dependencies and constraints, bundles can handle these relationships effectively.
4. **Familiarity**: If your sales team is already using CPQ effectively for other products, it might be easier to maintain consistency by adding configurable products to CPQ.

### Product Configurator with Revenue Cloud's CML
1. **Complex Configurations**: If the configurable product has extensive relationships, dependencies, and custom pricing rules, the flexibility of CML can be beneficial. It is designed for complex products with many intertwined options.
2. **Dynamic Pricing**: CML allows pricing models to be flexible and can adapt to highly intricate logic that may not be easily handled by traditional CPQ bundles.
3. **Decision Tables**: For highly configurable products, using decision tables in CML can streamline and automate pricing based on combinations of options, potentially leading to better pricing strategies.
4. **Future-Proofing**: Since you mentioned that CML and decision tables are seen as the future, adopting this technology might align with long-term strategic goals and could support further enhancements more efficiently compared to traditional bundling.

### Recommendation
If your client's setup is highly complex with many interdependencies among options, and the priority is on future-proofing the pricing strategy, utilizing the Product Configurator pattern with CML-based pricing could be the better choice. This allows for more robust and flexible configurations that can grow with the business.

However, if the configurations can be easily managed and used within a simpler structure, and the team is already comfortable using Salesforce CPQ for other products, sticking with the CPQ bundle could provide a quicker and possibly more efficient implementation.

In conclusion, assess the complexity of the options, the expertise of the team, and the future goals of the business to make the best decision.

The Skeptical CFO and their response:



When deciding between using Salesforce CPQ bundles with options or a Product Configurator pattern using Revenue Cloud's CML-based pricing for a manufacturing client dealing with configurable industrial equipment, consider the following aspects related to TCO, license cost, ROI timeline, and risk:

### Salesforce CPQ Bundles with Options
1. **TCO**: Salesforce CPQ typically has a higher total cost of ownership due to licensing fees and additional configuration costs.
2. **License Cost**: CPQ products usually require a separate license that can be costly, depending on the number of users and complexity.
3. **ROI Timeline**: The implementation time for CPQ can be significant due to the complexity of setting up bundles, which may delay ROI.
4. **Risk**: There's a risk of underestimating the complexity of setting up and maintaining the bundles, which can lead to attrition if staff are not adequately trained.

### Product Configurator Pattern using Revenue Cloud's CML-based Pricing
1. **TCO**: This approach can potentially lower TCO by utilizing existing Salesforce licenses and reducing the need for complex setup.
2. **License Cost**: If the client already has access to Revenue Cloud, this may reduce incremental license costs, thus being more economical.
3. **ROI Timeline**: A more streamlined setup could lead to quicker implementation, leading to faster realization of ROI.
4. **Risk**: Using a configurator may introduce less complexity and allow for a more seamless integration into existing sales processes, but it still requires careful planning for customizations.

### Recommendation:
- **If configuration flexibility and speed to market are key priorities, consider the Product Configurator pattern with CML-based pricing**, as it may offer a lower TCO and faster ROI.
- **If deep integration with the overall sales process and complex quote management is critical**, the CPQ bundle might be more suitable despite higher costs and longer ROI timelines.

Ultimately, weigh the specific needs of your manufacturing client against the financial aspects and risk tolerance associated with each option.

The End-User Sales Rep and their response:



For a manufacturing client selling configurable industrial equipment with over 200 options, it would be more suitable to model this as a Salesforce CPQ bundle with options rather than using Revenue Cloud's CML-based pricing. 

**Reasons to Choose CPQ Bundle:**

1. **Complexity Management:** Salesforce CPQ is designed to handle complex product configurations effectively. A bundle allows you to group options logically, making it easier for users to navigate and select configurations without overwhelming them with choices.

2. **User Experience:** CPQ's guided selling features help simplify the process for sales representatives, reducing the training burden and making it easier for them to generate quotes quickly.

3. **Click-to-Quote Efficiency:** Since your client cares about clicks-to-quote, using CPQ bundles can streamline the configuration process, ensuring that the generation of quotes happens swiftly and efficiently.

4. **Dynamic Pricing:** CPQ handles dynamic pricing based on selected options, which aligns well with the need for a responsive quoting system in a highly configurable environment.

5. **Visibility and Control:** With CPQ bundles, there’s better control over the product hierarchy and dependencies, ensuring that sales reps only see valid combinations, which can prevent configuration errors.

Overall, CPQ's structured approach to managing complex product configurations likely aligns better with your client's operational goals and pain points than a more flexible but potentially cumbersome CML-based pricing model.

Token usage for each actor:
CPQ Purist: 522 tokens
The Revenue Cloud Evangelist: 572 tokens
The Skeptical CFO: 477 tokens
The End-User Sales Rep: 349 tokens


In [38]:
#Synthesizer user

system_prompt_final_user = f"You are a non-biased synthesizer who has to make a decision based on the input of 4 different actors. Your job is to analyze the perspectives of all the actors and make a final decision based on their input. The 4 actors are {system_prompt_actors} and each has their answer with their respective responses and biases. Provide your final view on the matter. \n\n"

final_user_prompt = user_prompt + "\n\n" + "Here are the perspectives of the 4 actors:\n\n"
for actor, response in response_dict.items():
    final_user_prompt += f"{actor}:\n{response}\n\n"  

final_response = openai_client.chat.completions.create(model="gpt-4o-mini", messages=[
    {"role": "system", "content": system_prompt_final_user},
    {"role": "user", "content": final_user_prompt}
])
print("Final synthesis response:\n")
display(Markdown(final_response.choices[0].message.content))
total_token_usage = final_response.usage.total_tokens + sum(actor_token_usage.values())
print(f"Total token usage for the entire process: {total_token_usage} tokens")
print(f"Token usage for final synthesis response: {final_response.usage.total_tokens} tokens")

Final synthesis response:



Given the diverse perspectives from the actors regarding whether to model a manufacturing client's configurable industrial equipment as a Salesforce CPQ bundle with options or using the Product Configurator pattern through Revenue Cloud's CML-based pricing, let’s summarize their views and weigh the pros and cons to arrive at a coherent conclusion.

### Summary of Perspectives:

1. **CPQ Purist**: Advocates for CPQ bundles primarily due to their familiarity, ease of management, and the structured approach they provide for simpler configurations, suggesting they maintain performance and efficiency in quoting.

2. **Revenue Cloud Evangelist**: Promotes the flexibility and scalability of the Product Configurator with CML-based pricing, emphasizing its capability to handle complex configurations and dynamic pricing strategies, which may serve future growth better.

3. **Skeptical CFO**: Focuses on the financial implications—total cost of ownership (TCO), ROI timeline, and risk. The CFO suggests that while CPQ tends to have higher setup costs, the Product Configurator could lower financial barriers and facilitate a quicker ROI.

4. **End-User Sales Rep**: Prefers the CPQ bundle approach for the ease of use and efficiency it provides in generating quotes while minimizing clicks and reducing the burden of training, indicating user experience is critical.

### Weighing the Inputs:

- **User Experience and Operational Efficiency**: The emphasis on minimizing clicks-to-quote and reducing training burdens by the End-User Sales Rep and the general usability highlighted by both the CPQ Purist and Revenue Cloud Evangelist suggests that ease of use is paramount in this scenario. 

- **Complexity**: If the configurations are indeed complex with many interdependencies, as suggested by the Revenue Cloud Evangelist, the CML approach may offer necessary flexibility. However, both the CPQ Purist and the End-User Rep emphasize manageable complexity using bundles, which is critical for user efficiency.

- **Financial Considerations**: The CFO’s perspective on TCO and ROI highlights potential financial pitfalls of adopting CPQ if complexity might lead to higher licensing and implementation costs. However, the quick ROI and better cost management of the Product Configurator via existing licenses is appealing.

### Conclusion and Recommendation:

Choosing **Salesforce CPQ Bundles with Options** seems to be the most prudent approach for the manufacturing client, focusing on:

- **User Experience**: The simplicity in quoting and training aligns well with sales team needs, minimizing disruption and operational friction.
- **Efficiency in Handling Complex Configurations**: While it may not offer the same flexibility as CML, the structured approach of bundles can adequately manage complexity without overwhelming users.
- **Alignment with Established Practices**: Given the team’s experience and familiarity with CPQ, it enhances confidence in implementation and ongoing management.

While it is acknowledged that the Product Configurator pattern provides more flexibility for future growth and complex needs, the immediate operational realities highlighted by the End-User Rep and the financial perspectives from the CFO point towards a preserved focus on user efficiency and cost-effectiveness.

Therefore, the recommendation is to **model the equipment as a Salesforce CPQ bundle with options**, keeping the potential of future needs in mind, allowing room for evolution without derailing current operations.

Total token usage for the entire process: 4462 tokens
Token usage for final synthesis response: 2542 tokens
